# Dynamic Pricing Optimization - Phase 3: Demand Prediction Models

This notebook trains and compares three regression models to predict weekly demand (sales).

## Phase 3 Goals
1. Load Phase 2 engineered dataset
2. Perform leakage-safe time-based train/test split
3. Train Linear Regression, Random Forest, Gradient Boosting
4. Evaluate MAE, RMSE, R2
5. Select and save the best model
6. Visualize model performance

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

sys.path.append(os.path.abspath('../src'))

from model_training import (
    load_phase3_data,
    train_three_models,
    evaluate_models,
    get_best_model_name,
    save_phase3_artifacts
)
from feature_engineering import split_scale_for_modeling

print('Phase 3 imports loaded successfully.')

## 1. Load Data

In [ ]:
df = load_phase3_data('../data/walmart_ml_ready.csv')
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
print('Columns in Phase 3 dataset:')
for i, c in enumerate(df.columns, 1):
    print(f'{i:2}. {c}')

## 2. Leakage-Safe Split and Scaling

In [ ]:
X_train, X_test, y_train, y_test, scaler, scaled_cols = split_scale_for_modeling(
    df,
    target_col='Weekly_Sales',
    date_col='Date',
    test_size=0.2
)

print(f'Train shape: {X_train.shape}')
print(f'Test shape: {X_test.shape}')
print(f'Scaled feature columns: {len(scaled_cols)}')

## 3. Train Models

In [ ]:
models = train_three_models(X_train, y_train, random_state=42)
print('Models trained:')
print(list(models.keys()))

## 4. Evaluate Models

In [ ]:
metrics_df, predictions = evaluate_models(models, X_test, y_test)
metrics_df

In [ ]:
best_model_name = get_best_model_name(metrics_df)
print(f'Best model (by RMSE): {best_model_name}')

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=metrics_df, x='Model', y='RMSE', hue='Model', legend=False, palette='Set2')
plt.title('Model Comparison by RMSE', fontsize=14, fontweight='bold')
plt.ylabel('RMSE')
plt.xlabel('Model')
plt.tight_layout()
plt.savefig('../output/phase3_model_rmse_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: ../output/phase3_model_rmse_comparison.png')

## 5. Prediction Diagnostics

In [ ]:
y_pred_best = predictions[best_model_name]

plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred_best, alpha=0.35)
mn = min(y_test.min(), y_pred_best.min())
mx = max(y_test.max(), y_pred_best.max())
plt.plot([mn, mx], [mn, mx], 'r--', linewidth=2)
plt.xlabel('Actual Weekly Sales')
plt.ylabel('Predicted Weekly Sales')
plt.title(f'Actual vs Predicted ({best_model_name})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../output/phase3_actual_vs_predicted.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: ../output/phase3_actual_vs_predicted.png')

In [ ]:
errors = y_test - y_pred_best

plt.figure(figsize=(10, 5))
sns.histplot(errors, bins=50, kde=True)
plt.title(f'Residual Distribution ({best_model_name})', fontsize=14, fontweight='bold')
plt.xlabel('Residual (Actual - Predicted)')
plt.tight_layout()
plt.savefig('../output/phase3_residual_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: ../output/phase3_residual_distribution.png')

## 6. Save Best Model and Metrics

In [ ]:
artifacts = save_phase3_artifacts(
    metrics_df=metrics_df,
    models=models,
    best_model_name=best_model_name,
    scaler=scaler,
    feature_columns=list(X_train.columns),
    out_models_dir='../models',
    out_output_dir='../output',
)

print('Phase 3 artifacts saved:')
for k, v in artifacts.items():
    print(f'- {k}: {v}')

## Phase 3 Summary

- Trained and compared Linear Regression, Random Forest, and Gradient Boosting
- Evaluated with MAE, RMSE, R2 on a time-based holdout set
- Saved best-performing model and preprocessing artifacts

Next: Phase 4 Price Elasticity Analysis